# Module 4 — Add AgentCore Observability

Your agent is **built** (Module 1) and **deployed** (Module 2). The last rung makes it **observable**:
you'll *see* exactly what every request does — which tools it called, how many tokens it used, how long
each step took, and where it failed — in the **Amazon CloudWatch GenAI Observability** dashboard.

### What it takes (no agent code changes)

Observability adds **zero agent code** — the agent here is byte-identical to Module 2. It's three
operational switches plus a look at the dashboard:

| Step | What happens | Where |
|------|--------------|-------|
| **1. Transaction Search** | Account-level switch — makes spans searchable in `/aws/spans` | one-time, account (a script) |
| **2. Deploy** | The agent's image already runs under `opentelemetry-instrument` (ADOT), so it **emits** OTEL spans | `agentcore deploy` |
| **3. Runtime Tracing toggle** | Per-runtime switch — **delivers** the agent's spans to CloudWatch | console, per deployed agent |
| **4. Invoke + view** | Generate traffic (with a session id) and read the trace waterfall | console dashboard |

> Two distinct things have to be true: the agent must **emit** spans (Step 2 — the container's
> `opentelemetry-instrument` wrapper) *and* the runtime must be told to **deliver** them (Step 3 — the
> Tracing toggle). Account-level Transaction Search (Step 1) makes them searchable.

## Why observability?

When an agent runs autonomously through a multi-step loop, "the answer looks wrong" tells you almost
nothing. Observability turns the black box into a glass box:

- **Trace waterfall** — every step and tool call, for debugging
- **Token & cost** — what each invocation costs
- **Session correlation** — group everything by user/session for support
- **Latency & errors** — find the slow step, see where it failed

A production agent you can't see into is one you can't trust or operate.

## What you'll see in a trace (reading, not writing)

The runtime emits spans following **OpenTelemetry GenAI semantic conventions**, which is what lets the
CloudWatch dashboard render them as an agent trace:

```
invoke_agent cos                         ← top-level span for one request
├── gen_ai.operation.name = "invoke_agent"
├── gen_ai.usage.input_tokens / output_tokens
├── session.id = "<your session id>"     ← groups invocations in the same conversation
└── execute_tool <name>                  ← one child span per tool the agent used
    ├── Bash  (ran a script)
    ├── Read  (read financial_data / CLAUDE.md)
    └── Task  (delegated to a subagent)
```

You **read** these conventions to interpret a trace. You do **not** hand-write spans — the AgentCore
runtime does the instrumentation for you. (That's the modern, recommended path; older examples that
hand-built GenAI spans are no longer necessary for runtime-hosted agents.)

## Setup

Run the cell below to install all dependencies (Node.js, AgentCore CLI, Python packages) and
register the Jupyter kernel. After it completes, **select the `module-4-observability` kernel** from the
kernel picker (top-right) and continue with the rest of the notebook.

In [ ]:
!bash setup.sh

In [8]:
!curl -fsSL https://rpm.nodesource.com/setup_20.x | sudo bash -
!sudo yum install -y nodejs-20.20.2
!sudo npm install -g @aws/agentcore@0.17.0
!cd agentcore/cdk && npm ci
!agentcore --version

2026-06-06 10:47:36 - Cleaning up old repositories...
2026-06-06 10:47:36 - Old repositories removed
2026-06-06 10:47:36 - Supported architecture: aarch64
2026-06-06 10:47:36 - Added N|Solid repository for LTS version: 20.x
2026-06-06 10:47:36 - dnf available, updating...
Node.js Packages for Linux RPM based distros -  166 kB/s | 3.0 kB     00:00    
Metadata cache created.
N|Solid Packages for Linux RPM based distros -  169 kB/s | 3.0 kB     00:00    
Metadata cache created.
2026-06-06 10:47:37 - Repository is configured and updated.
2026-06-06 10:47:37 - You can use N|solid Runtime as a node.js alternative
2026-06-06 10:47:37 - To install N|solid Runtime, run: dnf install nsolid -y
2026-06-06 10:47:37 - Run 'dnf install nodejs -y' to complete the installation.
Package nodejs-2:20.20.2-1nodesource.aarch64 is already installed.
Dependencies resolved.
Nothing to do.
Complete!
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹

### Install dependencies & register kernel

In [3]:
import subprocess, shutil
from pathlib import Path

module_dir = Path.cwd()
module_name = module_dir.name  # e.g. "module-4-observability"

# 1) .env
if not (module_dir / ".env").exists():
    shutil.copy(".env.example", ".env")
    print("✅ created .env from .env.example — edit it if you need a different model/region")
else:
    print("✅ .env already exists")

# 2) uv sync
subprocess.run(["uv", "sync"], check=True, cwd=module_dir)
print("✅ uv sync done")

# 3) Register Jupyter kernel
venv_python = module_dir / ".venv" / "bin" / "python"
subprocess.run([
    str(venv_python), "-m", "ipykernel", "install",
    "--user", "--name", module_name, "--display-name", module_name,
], check=True)
print(f"✅ Kernel registered: {module_name}")

✅ .env already exists
✅ uv sync done


Resolved 164 packages in 0.61ms
Checked 142 packages in 83ms


Installed kernelspec module-4-observability in /home/participant/.local/share/jupyter/kernels/module-4-observability
✅ Kernel registered: module-4-observability


### Generate deployment target (`aws-targets.json`)

In [22]:
import json, os, subprocess

# --- Resolve region: single source of truth for the whole notebook ---
# Priority: AWS_REGION env (set by Workshop Studio) → AWS CLI config → fallback
_cli_region = subprocess.run(
    ["aws", "configure", "get", "region"], capture_output=True, text=True
).stdout.strip()
REGION = os.environ.get("AWS_REGION") or _cli_region or "us-west-2"

account_id = subprocess.run(
    ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
    capture_output=True, text=True,
).stdout.strip()

# Generate aws-targets.json from the resolved values
targets = [
    {
        "name": "default",
        "description": "Workshop deployment target (auto-generated).",
        "account": account_id,
        "region": REGION,
    }
]

with open("agentcore/aws-targets.json", "w") as f:
    json.dump(targets, f, indent=2)

print(f"✅ agentcore/aws-targets.json written:")
print(f"   account: {account_id}")
print(f"   region:  {REGION}")

✅ agentcore/aws-targets.json written:
   account: 361691913159
   region:  us-east-1


In [ ]:
from dotenv import load_dotenv
load_dotenv()

import boto3

print(f"Deploy region: {REGION}")

# Confirm AWS identity
try:
    who = boto3.client("sts").get_caller_identity()
    print(f"✅ AWS identity: {who['Arn']}")
except Exception as e:
    print(f"⚠️  AWS credentials not usable: {e}")

## Step 1 — Enable CloudWatch Transaction Search (one-time, account-level)

This is the **only** genuinely new setup in Module 4. Transaction Search is what makes the agent's
OpenTelemetry spans searchable in CloudWatch (they land in the `/aws/spans` log group). It's an
account-level switch — you do it once, not per deployment.

The helper is **idempotent** (safe to re-run): it checks the current state and only changes what's
missing.

In [5]:
result = subprocess.run(
    ["python", "scripts/enable_transaction_search.py", "--region", REGION],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)
# Note: after first enabling, allow ~10 minutes before spans are fully searchable.

Enabling CloudWatch Transaction Search in us-east-1 (account 361691913159)…
  • log resource policy 'TransactionSearchAccess' already present
  • trace segment destination already CloudWatchLogs/ACTIVE

✅ Already enabled — Destination=CloudWatchLogs, Status=ACTIVE.
Note: it can take ~10 minutes for spans to become searchable in /aws/spans.



## Step 2 — Deploy the (already-observable) agent

This is the *same* deploy as Module 2 — nothing extra. Because `enableOtel: true` and ADOT are already in
the config/image, the deployed agent is instrumented automatically. Look at the runtime config:

In [6]:
cfg = json.load(open("agentcore/agentcore.json"))
rt = cfg["runtimes"][0]
print(json.dumps({
    "name": rt["name"], "build": rt["build"], "protocol": rt["protocol"],
    "instrumentation": rt.get("instrumentation"),
}, indent=2))
# enableOtel: true  → the AgentCore runtime wraps the agent with opentelemetry-instrument on deploy.

{
  "name": "cos",
  "build": "Container",
  "protocol": "HTTP",
  "instrumentation": {
    "enableOtel": true
  }
}


Deploy from a **terminal** (builds the image in the cloud via CodeBuild, ~several minutes):

```bash
agentcore deploy -y
agentcore status        # confirm the runtime is READY — note its agent id / ARN for the next step
```

The container's `CMD` runs the agent under **`opentelemetry-instrument`** (from `aws-opentelemetry-distro`),
so once deployed the agent **emits** OTEL spans automatically. Next we tell the runtime to **deliver** them.

In [6]:
!agentcore deploy -y

✓ Load deployment target
⠋ Validate project...(node:426560) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
✓ Validate project
✓ Build CDK project...
✓ Synthesize CloudFormation...
✓ Check bootstrap status...
✓ Check stack status...
⠋ Deploy to AWS...(node:426560) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
✓ Deploy to AWS
✓ Persi

In [3]:
!agentcore status

(node:419144) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
AgentCore Status (target: default, us-west-2)

Agents
  cos: Deployed - Runtime: READY (arn:aws:bedrock-agentcore:us-west-2:3616919131
59:runtime/cosobserve_cos-JzqPQ2BNPk)
  URL: https://bedrock-agentcore.us-west-2.amazonaws.com/runtimes/arn%3Aaws%3Abe
drock-agentcore%3Aus-west-2%3A361691913159%3Aruntime%2Fcosobserve_cos-JzqPQ2BNPk
/invocations


## Step 3 — Enable Tracing on the runtime (one toggle, in the console)

A freshly deployed AgentCore runtime **emits** spans but doesn't **deliver** them to CloudWatch until you
turn on its **Tracing** toggle. This is a per-runtime switch and is done in the console:

1. Open the **AgentCore → Agent Runtime** page:
   `https://console.aws.amazon.com/bedrock-agentcore/agents`
2. Select your agent (the one you just deployed — name starts with `cos`).
3. In the **Tracing** pane, choose **Edit**, toggle **Enable**, and **Save**.

> Once enabled, the agent's spans flow to the `aws/spans` log group and show up in the GenAI
> Observability dashboard. You only do this once per runtime.

*(Why a manual step? For AgentCore **runtime** resources, enabling trace delivery is a console action —
there isn't a public CLI/`agentcore.json` field for it yet. The account-level Transaction Search and the
container's OTEL emission are both automated above; this toggle is the one click that isn't.)*

## Step 4 — Generate traffic (with a session id)

Invoke the deployed agent a couple of times, passing a **session id**. The session id is what groups
related invocations together in the dashboard's *Sessions* view.

```bash
agentcore invoke --session-id "m4-observability-demo-session-001" "What is our current runway and cash position?"
agentcore invoke --session-id "m4-observability-demo-session-001" "If we hire 10 engineers, how does that change?"
```

In [31]:
!agentcore invoke --session-id "m4-observability-demo-session-001" "What is our current runway and cash position?"

⠋ Invoking agent...(node:386208) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
Here's a summary of our current runway and cash position:

---

## 💰 Cash Position & Runway Summary

| Metric | Value |
|--------|-------|
| **Cash in Bank** | $10M (from Series A, closed Jan 2024) |
| **Gross Monthly Burn** | ~$525K (as of June 2024, trending up with hiring) |
| **Monthly Revenue** | ~$290K (June 2024) |
| **Net Monthly Burn** | ~$235K (gross burn minus revenue) |
| **Runway (Gross)** | ~20 months |
| **Runway (Net, at current trajectory)** | ~42 months |
| **Daily Burn Rate** | ~$16.7K |
| **Quarterly Burn (Gross)** | ~$1.5M |

In [7]:
!agentcore invoke --session-id "m4-observability-demo-session-001" "If we hire 10 engineers, how does that change?"

⠋ Invoking agent...(node:431109) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
Use the financial-analyst subagent to analyze the budget impact of: Analyze the financial impact of hiring 10 engineers on our burn rate and runway. Use our current financials: $10M cash in bank, $500K monthly burn, 20 months runway. Compensation benchmarks: Senior Engineer $180K-$220K + equity, Junior Engineer $100K-$130K. Assume a mix of 6 senior and 4 junior engineers. Include recruiting costs, ramp-up time, and impact on revenue per employee.

Provide a comprehensive analysis including:
1. Total cost (one-time and recurring)
2. Impact on mon

In [13]:
# You can also invoke from the notebook:
SESSION_ID = "m4-observability-demo-session-001"
res = subprocess.run(
    ["agentcore", "invoke", "--session-id", SESSION_ID, "What is our current runway?"],
    capture_output=True, text=True,
)
print((res.stdout or res.stderr)[-1500:])

### Baseline Calculation
| Metric | Value |
|--------|-------|
| **Cash at Series A** | $10M |
| **Gross Monthly Burn** | ~$500K–$525K |
| **Gross Runway** | 20 months (from Jan 2024) |

### The Better Story: Net Burn Is Improving

The trend is strongly positive as revenue scales:

| Month | Gross Burn | Revenue | **Net Burn** |
|-------|-----------|---------|-------------|
| Jan 2024 | $450K | $180K | $270K |
| Mar 2024 | $490K | $210K | $280K |
| Jun 2024 | $525K | $290K | **$235K** |

Net burn dropped **13%** from January to June 2024 even as we added headcount, because revenue is growing faster (~15% MoM) than costs.

### Revenue Trajectory
Our ARR forecast projects growth from **$2.4M → $5.5M** by December 2024. At that pace, monthly revenue (~$460K) would nearly cover gross burn, dramatically extending runway or reaching cash-flow breakeven.

### ⚠️ Key Caveat
The original 20-month runway (expiring ~Sept 2025) assumed **zero revenue growth**. Given our trajectory, effective runwa

## Step 5 — View the traces

**In the console (the main event):** open the GenAI Observability dashboard — it has **Agents**,
**Sessions**, and **Traces** views. Pick your agent, drill into a session, and open a trace to see the
span waterfall (tool calls, token usage, latency).

```
https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability
```

(Replace `{REGION}`. Allow ~2–10 minutes after invoking for spans to be indexed.)


In [ ]:
import time

print(f"GenAI dashboard: "
      f"https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability")

# Peek at /aws/spans for spans from our session (best-effort; indexing can lag a few minutes).
logs = boto3.client("logs", region_name=REGION)

def find_spans(session_id, attempts=10, delay=30):
    query = (
        "fields @timestamp, @message "
        f"| filter @message like '{session_id}' "
        "| sort @timestamp desc | limit 20"
    )
    for i in range(attempts):
        try:
            start = logs.start_query(
                logGroupName="aws/spans",
                startTime=int(time.time()) - 3600,
                endTime=int(time.time()),
                queryString=query,
            )["queryId"]
        except Exception as e:
            print(f"  ⚠️  Could not query /aws/spans: {e}")
            print("  → Check the console dashboard directly (link above).")
            return False
        # poll this query
        while True:
            r = logs.get_query_results(queryId=start)
            if r["status"] in ("Complete", "Failed", "Cancelled"):
                break
            time.sleep(2)
        rows = r.get("results", [])
        if rows:
            print(f"✅ Found {len(rows)} span(s) for session '{session_id}':")
            for row in rows[:5]:
                d = {f["field"]: f["value"] for f in row}
                print(f"  • {d.get('@timestamp', '?')}")
            return True
        print(f"  …no spans yet (attempt {i+1}/{attempts}); waiting {delay}s for indexing")
        time.sleep(delay)
    print("⚠️  No spans found after polling — check the console dashboard (link above).")
    print("   (Spans can take up to ~10 min to become searchable in Logs Insights.)")
    return False

find_spans(SESSION_ID)

## Step 6 — Cleanup

Tear down the runtime to avoid ongoing charges (same as Module 2). Run in a **terminal**:

```bash
agentcore remove agent --name cos
agentcore deploy -y          # applies the removal → destroys the runtime/stack
```

> **Transaction Search stays enabled** — it's an account-level, one-time setting, not per-deployment, and
> there's no charge for leaving it on at low/no indexing sampling.

In [19]:
!agentcore remove agent --name cos

{"success":true,"resourceType":"agent","resourceName":"cos","message":"Removed agent 'cos'","note":"Your agent app source code has not been modified. Deploy with `agentcore deploy` to apply your removal changes to AWS."}


In [20]:
!agentcore deploy -y

✓ Load deployment target
✓ Validate project...
⠋ Validate AWS credentials...(node:344412) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
✓ Validate AWS credentials
✓ Build CDK project...
⠋ Synthesize CloudFormation...(node:344412) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
✓ Synthesize CloudFormation
✓ Check bootstrap status...


## Key takeaways

- AgentCore Runtime **auto-instruments** your agent with OpenTelemetry — observability needs **no agent
  code change** (it was already on from Module 2's `enableOtel` + ADOT).
- The one new step is **enabling CloudWatch Transaction Search** (account-level, one-time).
- Traces follow **GenAI semantic conventions** (`gen_ai.*`, `session.id`), which is why the **GenAI
  Observability dashboard** can render the agent's trace waterfall, tokens, and tool calls.
- Pass a **session id** on invoke to correlate a conversation in the dashboard.

🎉 You've climbed the whole ladder: **built → deployed → observable.** (Memory — Module 3 — is the
remaining rung when you're ready.)